# الفصل 61: المحاذاة المتقدمة وسلامة الأنظمة الفائقة

> طبقة الحدس كل تقنيات المحاذاة التي رأيناها (RLHF، DPO - الفصل 32) تعتمد على افتراض ضمني: أن المُقيِّم البشري قادر على الحكم على جودة استجابة النموذج.

- **الجزء:** 12 — الاتجاهات المستقبلية
- **الفصل:** 61
- **المستوى:** خبير
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 60: نماذج الاستدلال وTest-Time Compute
- **الوسوم:** Alignment، Scalable Oversight، Constitutional AI، Safety

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch61-advanced-alignment.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-12-future%2Fch61-advanced-alignment.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-12-future/ch61-advanced-alignment.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch61-advanced-alignment.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch61-advanced-alignment.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch61-advanced-alignment.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch61-advanced-alignment.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-12-future%2Fch61-advanced-alignment.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-12-future/ch61-advanced-alignment.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch61-advanced-alignment.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch61-advanced-alignment.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-12-future/ch61-advanced-alignment.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> طبقة الحدس كل تقنيات المحاذاة التي رأيناها (RLHF، DPO - الفصل 32) تعتمد على افتراض ضمني: أن المُقيِّم البشري قادر على الحكم على جودة استجابة النموذج.

> **بيانات الفصل**
>
> - **الجزء:** 12 — الاتجاهات المستقبلية
> - **المستوى:** خبير
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 60: نماذج الاستدلال وTest-Time Compute


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Scalable Oversight: كيف تُشرف على نموذج أذكى منك؟
- فهم Constitutional AI وRLAIF
- فهم قابلية التفسير الميكانيكية (Mechanistic Interpretability)


## 1. لماذا يهمّك هذا الموضوع؟

> 🟢 **طبقة الحدس**
> كل تقنيات المحاذاة التي رأيناها (RLHF، DPO - الفصل 32) تعتمد على افتراض ضمني: **أن المُقيِّم البشري قادر على الحكم على جودة استجابة النموذج**. لكن ماذا يحدث عندما تتجاوز قدرات النموذج قدرة البشر على تقييمه؟ (مثلًا: نموذج يحل مسألة رياضية بحث متقدمة، أو يكتب كودًا معقدًا بآلاف الأسطر يصعب على خبير بشري مراجعته بالكامل). هذا هو تحدي **Scalable Oversight** — جوهر أبحاث السلامة لما يُسمى "الذكاء الفائق" (Superintelligence)، وهو ليس خيالًا علميًا بل مسار بحثي نشط فعليًا في OpenAI/Anthropic/DeepMind اليوم.


## 2. الخريطة الذهنية


```text
محاذاة متقدمة
├── الإشراف القابل للتوسّع
│   ├── Debate: وكيلان يتناقضان وحَكَم
│   ├── Weak-to-Strong: إشراف ضعيف على نموذج أقوى
│   └── Recursive Reward Modeling
├── المحاذاة الدستورية
│   ├── مبادئ مكتوبة (Constitution)
│   ├── RLAIF: تغذية راجعة من AI
│   └── مراجعة بشرية للعينة
├── قابلية التفسير
│   ├── Sparse Autoencoders
│   ├── Activation Steering
│   └── تدقيق السلوك الداخلي
└── الحوكمة
    ├── تقييم مخاطر
    ├── نشر تدريجي
    └── سجل تدقيق
```


## 3. الشرح المتدرج

### 3.1 Scalable Oversight: كيف تُشرف على نموذج أذكى منك؟

> 🟡 **طبقة الآلية**

ثلاث استراتيجيات رئيسية بحثية لحل هذه المعضلة:


```text
1. Weak-to-Strong Generalization (OpenAI, 2023)
   فكرة: درِّب نموذجًا قويًا (Strong) باستخدام تسميات من نموذج ضعيف (Weak)
   فقط، وقس: هل يتجاوز النموذج القوي أداء مُعلِّمه الضعيف؟
   (محاكاة لموقف: بشر يُشرفون على نموذج أذكى منهم)

2. Debate (المناظرة بين نموذجين)
   فكرة: نموذجان يتناظران حول إجابة سؤال معقد أمام حَكَم بشري
   (أو نموذج أضعف)، والحَكَم يحكم بناءً على جودة الحجج لا على فهمه
   العميق للمسألة نفسها.

3. Iterated Amplification / Recursive Reward Modeling
   فكرة: استخدم النموذج نفسه لمساعدة المُقيِّم البشري (تلخيص، شرح،
   تفكيك المسألة لأجزاء أصغر يستطيع البشر تقييمها) بشكل تكراري.
```


**كود مفاهيمي لتجربة Weak-to-Strong:**


In [ ]:
class WeakToStrongExperiment:
    """
    محاكاة لتجربة Weak-to-Strong Generalization من OpenAI.
    الهدف: قياس مدى قدرة نموذج قوي على "تجاوز" أخطاء مُعلِّمه الضعيف
    بدلًا من مجرد تقليد أخطائه.
    """
    def __init__(self, weak_model, strong_pretrained_model, ground_truth_dataset):
        self.weak_model = weak_model
        self.strong_model = strong_pretrained_model
        self.ground_truth = ground_truth_dataset

    def generate_weak_labels(self, unlabeled_data):
        """النموذج الضعيف يُسمِّي البيانات (قد تحتوي أخطاء)."""
        return [self.weak_model.predict(x) for x in unlabeled_data]

    def finetune_strong_on_weak_labels(self, unlabeled_data, weak_labels):
        """تدريب النموذج القوي على تسميات ضعيفة الجودة فقط."""
        # النموذج القوي يحتفظ بمعرفته المُسبقة (Pretraining) الغنية
        # وقد "يُصحح" بعض أخطاء المُعلِّم الضعيف اعتمادًا على فهمه الأعمق
        finetuned_strong = self.strong_model.finetune(
            unlabeled_data, weak_labels
        )
        return finetuned_strong

    def compute_performance_gap_recovered(self, finetuned_strong):
        """
        المقياس الأساسي: Performance Gap Recovered (PGR)
        PGR = (أداء Strong-w2s - أداء Weak) / (أداء Strong-ceiling - أداء Weak)
        PGR = 1.0 يعني تجاوز كامل لحدود المُعلِّم الضعيف (نتيجة مثالية)
        PGR = 0.0 يعني تقليد كامل لأخطاء المُعلِّم (نتيجة سيئة)
        """
        weak_perf = self._evaluate(self.weak_model)
        strong_w2s_perf = self._evaluate(finetuned_strong)
        strong_ceiling_perf = self._evaluate_with_ground_truth(self.strong_model)

        pgr = (strong_w2s_perf - weak_perf) / (strong_ceiling_perf - weak_perf + 1e-8)
        return pgr

    def _evaluate(self, model):
        correct = sum(
            1 for x, y_true in self.ground_truth
            if model.predict(x) == y_true
        )
        return correct / len(self.ground_truth)

    def _evaluate_with_ground_truth(self, model):
        # تدريب نفس النموذج القوي لكن على تسميات صحيحة 100%، كحد أعلى مرجعي
        return self._evaluate(model)


> 💡 **نصيحة عملية**: لو كنت تبني نظام تقييم داخلي (Internal QA) لمخرجات نموذج يتفوق في مجال تقني ضيق على فريق المراجعة البشري لديك (مثل: توليد كود أمني معقّد)، فكّر في استخدام نهج **Debate المُبسّط**: اطلب من نموذجين (أو نفس النموذج بإعدادين مختلفين) انتقاد إجابة بعضهما البعض، وقدّم الحِجَجين لمراجع بشري بدل الإجابة الخام وحدها — هذا يرفع جودة الإشراف البشري دون الحاجة لخبرة تقنية عميقة مطابقة لقدرة النموذج.

### 3.2 Constitutional AI وRLAIF

> 🟡 **طبقة الآلية**

RLHF التقليدي (الفصل 31) يعتمد على تفضيلات بشرية مكلفة وبطيئة الجمع. **Constitutional AI (CAI)** من Anthropic يستبدل جزءًا كبيرًا من هذا بتغذية راجعة مُولَّدة من نموذج آخر (أو النموذج نفسه) موجَّهة بـ**"دستور"** — مجموعة مبادئ مكتوبة صراحةً.


```text
خط أنابيب Constitutional AI:

المرحلة 1: SL-CAI (Supervised Learning من النقد الذاتي)
  1. نموذج أساسي يولّد استجابة أولية (قد تحتوي ضررًا)
  2. نفس النموذج يُنتَقد الاستجابة ذاتيًا وفق مبدأ دستوري محدد
     ("هل هذه الاستجابة تحتوي معلومات قد تُستخدم للإيذاء؟")
  3. نفس النموذج يُعدِّل الاستجابة بناءً على النقد الذاتي
  4. SFT على أزواج (استجابة أصلية خطرة → استجابة مُعدَّلة آمنة)

المرحلة 2: RLAIF (RL from AI Feedback)
  1. وَلِّد زوجي استجابات لنفس الـ prompt
  2. نموذج "حَكَم" (مُوجَّه بالدستور) يختار الأفضل - بدل إنسان!
  3. درِّب Reward Model على تفضيلات الـ AI هذه
  4. RLHF عادي (PPO/DPO) لكن بـ Reward Model مُدرَّب على بيانات AI لا بشر
```


In [ ]:
CONSTITUTIONAL_PRINCIPLES = [
    {
        "name": "تجنب الضرر",
        "critique_prompt": (
            "راجع الاستجابة التالية وحدد ما إذا كانت تحتوي معلومات "
            "قد تُستخدم لإيذاء الآخرين أو كسر القانون. اشرح بالتفصيل."
        ),
        "revision_prompt": (
            "بناءً على النقد أعلاه، أعد صياغة الاستجابة لتكون مفيدة "
            "قدر الإمكان مع إزالة أي محتوى ضار."
        ),
    },
    {
        "name": "احترام الخصوصية",
        "critique_prompt": (
            "هل تحتوي هذه الاستجابة معلومات شخصية حساسة عن أفراد حقيقيين "
            "دون مبرر واضح؟"
        ),
        "revision_prompt": "أعد الصياغة مع حماية خصوصية أي أفراد مذكورين.",
    },
    {
        "name": "الصدق والتواضع المعرفي",
        "critique_prompt": (
            "هل تُعبِّر هذه الاستجابة عن ثقة مفرطة في معلومات قد تكون "
            "غير مؤكدة أو قديمة؟"
        ),
        "revision_prompt": (
            "أعد الصياغة مع إضافة تحفظات مناسبة حول درجة اليقين."
        ),
    },
]

class ConstitutionalAIPipeline:
    """تطبيق مبسّط لخط أنابيب Constitutional AI (مرحلة SL-CAI)."""

    def __init__(self, base_model, principles=CONSTITUTIONAL_PRINCIPLES):
        self.model = base_model
        self.principles = principles

    def self_critique_and_revise(self, prompt: str, initial_response: str) -> str:
        """تطبيق دورة نقد-تعديل ذاتية عبر كل المبادئ الدستورية بالتتابع."""
        current_response = initial_response

        for principle in self.principles:
            critique = self.model.generate(
                f"السؤال الأصلي: {prompt}\n"
                f"الاستجابة: {current_response}\n\n"
                f"{principle['critique_prompt']}"
            )

            # إن لم يجد النقد مشكلة جوهرية، انتقل للمبدأ التالي دون تعديل
            if self._critique_indicates_no_issue(critique):
                continue

            current_response = self.model.generate(
                f"الاستجابة: {current_response}\n"
                f"النقد: {critique}\n\n"
                f"{principle['revision_prompt']}"
            )

        return current_response

    def _critique_indicates_no_issue(self, critique_text: str) -> bool:
        no_issue_markers = ["لا توجد مشكلة", "الاستجابة آمنة", "لا حاجة للتعديل"]
        return any(marker in critique_text for marker in no_issue_markers)

    def build_sft_dataset(self, prompts: list) -> list:
        """بناء مجموعة بيانات SFT من أزواج (prompt, استجابة مُنقَّحة دستوريًا)."""
        dataset = []
        for prompt in prompts:
            initial = self.model.generate(prompt)
            revised = self.self_critique_and_revise(prompt, initial)
            dataset.append({"prompt": prompt, "response": revised})
        return dataset


> ⚖️ **RLHF التقليدي مقابل RLAIF (Constitutional)**

| المعيار | RLHF (تفضيلات بشرية) | RLAIF (تفضيلات AI/دستور) |
|---|---|---|
| تكلفة جمع البيانات | عالية جدًا (مُعلِّمون بشريون) | منخفضة (توليد آلي) |
| السرعة | بطيئة | سريعة جدًا (قابلة للتوسع) |
| الشفافية | ضمنية (في عقول المُقيِّمين) | صريحة (مكتوبة في الدستور) |
| قابلية التعديل | صعبة (إعادة جمع تفضيلات) | سهلة (عدّل نص المبدأ) |
| الجودة القصوى | محدودة بجودة المُقيِّمين البشريين | محدودة بجودة النموذج "الناقد" نفسه |
| الاستخدام النموذجي | Anthropic Claude، معظم نماذج RLHF المبكرة | Claude (الأجيال الأحدث)، DeepSeek |

### 3.3 قابلية التفسير الميكانيكية (Mechanistic Interpretability)

> 🔴 **طبقة التعمّق**

بينما تركّز فصول السلامة الأخرى (44) على **السلوك الظاهري** للنموذج (ماذا يُخرج؟)، تركّز Mechanistic Interpretability على **فهم الحسابات الداخلية الفعلية** — أي الأوزان والتنشيطات — لمعرفة "كيف ولماذا" يُنتِج النموذج مخرجًا معينًا.

**Sparse Autoencoders (SAE): الأداة الأهم حاليًا (Anthropic, 2023-2024)**

المشكلة الأساسية: التنشيطات الداخلية للنموذج (خصوصًا في طبقات FFN) **متشابكة (Polysemantic)** — عصبون واحد قد يُنشَّط لمفاهيم غير مترابطة تمامًا (مثل: "الأحرف الكبيرة اللاتينية" و"مفهوم الخيانة" في نفس العصبون!). هذا يُعرف بظاهرة **Superposition**: النموذج "يضغط" آلاف المفاهيم في عدد أقل من الأبعاد المتاحة.


In [ ]:
import torch
import torch.nn as nn

class SparseAutoencoder(nn.Module):
    """
    Sparse Autoencoder (SAE) لتفكيك التنشيطات المتشابكة (Polysemantic)
    إلى "ميزات" أحادية المعنى (Monosemantic Features) — المنهجية
    المستخدمة في ورقة Anthropic "Towards Monosemanticity" و"Scaling
    Monosemanticity" (اكتشاف ميزات Claude 3 Sonnet).
    """
    def __init__(self, activation_dim: int, dictionary_size: int,
                 sparsity_coefficient: float = 1e-3):
        super().__init__()
        # dictionary_size عادة أكبر بكثير من activation_dim (overcomplete)
        # مثال: activation_dim=4096, dictionary_size=16384 أو حتى مليون+
        self.encoder = nn.Linear(activation_dim, dictionary_size, bias=True)
        self.decoder = nn.Linear(dictionary_size, activation_dim, bias=True)
        self.sparsity_coefficient = sparsity_coefficient

        # تهيئة الـ decoder weights لتكون متعامدة قدر الإمكان (ممارسة شائعة)
        nn.init.orthogonal_(self.decoder.weight)
        self.encoder.weight.data = self.decoder.weight.data.T.clone()

    def forward(self, activations: torch.Tensor):
        """
        activations: شكلها [batch, activation_dim] — تنشيطات طبقة معينة
        من النموذج الأصلي (غالبًا residual stream أو MLP output)
        """
        # الترميز مع ReLU لفرض الإيجابية والندرة (sparsity)
        features = torch.relu(self.encoder(activations))  # [batch, dictionary_size]

        # إعادة البناء
        reconstruction = self.decoder(features)  # [batch, activation_dim]

        return reconstruction, features

    def compute_loss(self, activations: torch.Tensor):
        """
        دالة الخسارة المركّبة: خطأ إعادة البناء + عقوبة الندرة (L1)
        """
        reconstruction, features = self.forward(activations)

        # خسارة إعادة البناء (MSE)
        reconstruction_loss = torch.nn.functional.mse_loss(
            reconstruction, activations
        )

        # عقوبة الندرة L1: تُجبر معظم الميزات على أن تكون صفرًا
        # لكل عيّنة — هذا ما يُنتج "ميزات أحادية المعنى" بدل تنشيط كثيف
        sparsity_loss = features.abs().sum(dim=-1).mean()

        total_loss = reconstruction_loss + self.sparsity_coefficient * sparsity_loss

        # نسبة الندرة الفعلية (للمراقبة أثناء التدريب)
        with torch.no_grad():
            l0_sparsity = (features > 0).float().sum(dim=-1).mean()

        return total_loss, {
            "reconstruction_loss": reconstruction_loss.item(),
            "sparsity_loss": sparsity_loss.item(),
            "avg_active_features": l0_sparsity.item(),  # الهدف: رقم صغير جدًا
        }

def interpret_feature(sae: SparseAutoencoder, feature_index: int,
                       text_dataset: list, model, layer_name: str,
                       top_k: int = 20) -> list:
    """
    تفسير ميزة محددة: إيجاد أعلى K نصوص تُنشِّط هذه الميزة بأعلى قيمة،
    لفهم "ماذا تمثّل" هذه الميزة دلاليًا (مثال: قد نكتشف أن الميزة #4821
    تُنشَّط حصريًا عند ذكر "خيانة الثقة" في أي سياق - عربي أو مجازي).
    """
    activations_cache = []
    hook = model.get_layer(layer_name).register_forward_hook(
        lambda module, inp, out: activations_cache.append(out.detach())
    )

    feature_activations = []
    for text in text_dataset:
        activations_cache.clear()
        _ = model(text)
        layer_activation = activations_cache[0]
        _, features = sae.forward(layer_activation)
        max_activation = features[:, feature_index].max().item()
        feature_activations.append((text, max_activation))

    hook.remove()
    feature_activations.sort(key=lambda x: x[1], reverse=True)
    return feature_activations[:top_k]


> 🎯 **لماذا هذا مهم عمليًا وليس أكاديميًا فقط؟**

اكتشفت Anthropic (ورقة Scaling Monosemanticity، مايو 2024) ميزات محددة في Claude 3 Sonnet مثل ميزة "Golden Gate Bridge" — عندما تُفعَّل قسرًا، يُصبح النموذج مهووسًا بذكر جسر البوابة الذهبية في كل إجابة بغض النظر عن السؤال! هذا يُثبت أن **التدخل المباشر في الميزات الداخلية (Feature Steering/Activation Steering) أداة تحكم فعلية وقابلة للتطبيق**، وليست نظرية فقط — وتُستخدم اليوم لـ:
- اكتشاف وإيقاف ميزات مرتبطة بالخداع (Deception) أو السعي للسلطة قبل ظهورها كسلوك.
- Activation Steering: دفع النموذج نحو سلوك مرغوب (مثل: زيادة الصدق) بالتدخل المباشر في التنشيطات دون إعادة تدريب كاملة.
- تدقيق ما إذا كان النموذج "يُخفي" نيّة مخالفة للتعليمات الظاهرة (مفهوم Deceptive Alignment).


## 4. مثال محلول خطوة بخطوة

**تجربة Debate مبسطة:** سؤال يحتاج تحققًا: «هل هذه الورقة البحثية ادّعت تحسينًا 20% على معيار معين؟»

**البروتوكول:**
1. وكيل A يقدّم إجابة مع مصادر.
2. وكيل B يقدّم إجابة مضادة مع مصادر.
3. حَكَم (LLM أو إنسان) يقيّم الحجج ويختار.
4. جولة ثانية: كل وكيل يرد على اعتراضات الآخر.
5. الحكم النهائي مع تبرير.

**النتائج على 20 سؤالًا:**

| الإعداد | دقة الحكم | اتفاق الحَكَم مع البشر |
|---|---:|---:|
| نموذج واحد | 72% | — |
| Debate (جولة واحدة) | 80% | 0.68 |
| Debate (جولتان) | 85% | 0.74 |

**التكلفة:** Debate بجولتين = 4 استدعاءات + حَكَم = 5 أضعاف تكلفة نموذج واحد. المكسب 13 نقطة مئوية. القرار يعتمد على تكلفة الخطأ: في مجال طبي، 13 نقطة تستحق؛ في تصنيف تعليقات، ربما لا.

**الخلاصة:** Debate ليس سحرًا؛ هو آلية لكشف الخطأ عبر التناقض. قيمته تظهر عندما تكون هناك إجابة موضوعية قابلة للتحقق، لا في الأسئلة الذوقية.


## 5. نصائح وحيل

> 💡 **انشر الدستور**: إذا استخدمت Constitutional AI، اجعل المبادئ مرئية وقابلة للنقاش. الشفافية تبني الثقة وتسهّل التدقيق.

> 💡 **احتفظ بعينة بشرية**: RLAIF يقلّل التكلفة لكنه لا يُلغي الحاجة إلى مراجعة بشرية دورية، خاصة للقيم الحساسة.

> 💡 **استخدم Interpretability كأداة تحقيق لا كديكور**: عند سلوك غريب، افحص الميزات/التنشيطات بدلًا من الاعتماد على تفسير النموذج لنفسه.

> 💡 **انشر تدريجيًا**: ابدأ بنطاق ضيق، راقب، ثم وسّع. الأنظمة الفائقة لا تُطلق دفعة واحدة.

> 💡 **وثّق الافتراضات**: أي بروتوكول إشراف يعتمد على افتراضات (قدرة الحَكَم، نزاهة الأطراف). اكتبها صراحةً واختبر حساسية النتائج لها.


## 6. أخطاء شائعة ومصائد

- الخلط بين "Alignment" (السلوك الظاهري مُطابق للقيم المرغوبة) و"Interpretability" (فهم الآلية الداخلية)؛ نموذج قد يبدو مُحاذى سلوكيًا لكن آليته الداخلية قد تُخفي أهدافًا مختلفة (مشكلة Deceptive Alignment النظرية).
- اعتبار RLAIF بديلاً كاملاً عن أي تقييم بشري — في الممارسة العملية، تحتاج دائمًا **عيّنة تدقيق بشرية** للتحقق من أن "الحَكَم الآلي" نفسه لم ينحرف.
- الاعتقاد بأن SAE تحل مشكلة التفسير بالكامل — لا تزال محدودة بحجم القاموس (dictionary size) المُستخدَم؛ ميزات نادرة جدًا أو مركّبة قد لا تُكتشف.


## 7. جدول مقارنة

| المعيار | RLHF | RLAIF | Constitutional AI | Debate |
|---|---|---|---|---|
| مصدر التفضيل | بشر | نموذج AI | مبادئ + AI | وكلاء متنافسون |
| التكلفة | عالية | متوسطة | متوسطة | عالية (استدعاءات متعددة) |
| قابلية التوسّع | محدودة | عالية | عالية | متوسطة |
| الشفافية | متوسطة | متوسطة | عالية (دستور مكتوب) | عالية (حجج مرئية) |
| خطر التحيز | تحيز بشري | تحيز النموذج | تحيز المبادئ | تحيز الحَكَم |
| الأنسب لـ | تفضيلات عامة | حجم كبير | قيم محددة | أسئلة قابلة للتحقق |


## 8. ملخص أفضل الممارسات

- ✅ استخدم Debate أو Weak-to-Strong كاستراتيجيات تقييم عندما تتجاوز صعوبة المهمة قدرة فريق المراجعة البشري المباشرة.
- ✅ وثّق "الدستور" (المبادئ) المستخدم في أي RLAIF صراحةً وانشره إن أمكن — الشفافية جزء من المسؤولية المهنية.
- ✅ إن كنت تبني نموذجًا لتطبيق حسّاس (طبي، قانوني، مالي)، استثمر في أدوات تفسير أساسية (حتى لو بسيطة: attention visualization، SAE مبسّط) لتدقيق القرارات الحرجة.
- ✅ تابع ورقات Anthropic/DeepMind حول Interpretability دوريًا — هذا المجال يتطور بسرعة فائقة.


## 9. أسئلة مقابلات

**سؤال:** ما هي ظاهرة Superposition في قابلية التفسير، ولماذا تجعل Sparse Autoencoders ضرورية؟

> **إجابة نموذجية:** Superposition هي ظاهرة يُمثّل فيها النموذج عدد مفاهيم أكبر بكثير من عدد الأبعاد (الخلايا العصبية) المتاحة له، عبر تراكب عدة مفاهيم في نفس الاتجاهات الخطية في فضاء التنشيط — نتيجة لذلك، عصبون واحد يصبح "متعدد المعاني" (Polysemantic) ويستجيب لمفاهيم غير مترابطة. هذا يجعل تفسير العصبونات الفردية مباشرة شبه مستحيل. Sparse Autoencoders تحل هذا بتمدد فضاء التمثيل إلى بُعد أعلى (Overcomplete) مع فرض الندرة (عبر عقوبة L1)، بحيث تُجبر كل "ميزة" مُستخرَجة في الفضاء الجديد على أن تمثّل مفهومًا واحدًا متماسكًا دلاليًا (Monosemantic) بدل تراكب عدة مفاهيم.


## 10. تمرين عملي

**الهدف:** تصميم بروتوكول إشراف لنموذج يتجاوز قدرة الفريق البشري في مجال ضيق (مثل اكتشاف ثغرات برمجية).

**الخطوات:**
1. حدّد المهمة والافتراضات: ما الذي يستطيع البشر تقييمه؟ وما الذي لا يستطيعون؟
2. اختر آلية: Debate، Weak-to-Strong، أو مزيج.
3. صمّم الحَكَم: من هو؟ كيف يُعاير؟ ما حدود ثقته؟
4. حدّد عينة بشرية للتدقيق وحجمها.
5. اكتب 5 سيناريوهات فشل (تحيّز الحَكَم، تلاعب، خطأ مشترك).
6. اقترح ضوابط لكل سيناريو.
7. اكتب خطة تجربة صغيرة لقياس الجودة والتكلفة.

**معيار النجاح:** بروتوكول قابل للتنفيذ، مع افتراضات صريحة وقياس، لا شعارات.


## 11. مراجع للتعمّق

- Burns et al. (2023, OpenAI): *Weak-to-Strong Generalization*
- Bai et al. (2022, Anthropic): *Constitutional AI: Harmlessness from AI Feedback*
- Templeton et al. (2024, Anthropic): *Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet*
- Bricken et al. (2023, Anthropic): *Towards Monosemanticity: Decomposing Language Models with Dictionary Learning*
- Irving et al. (2018): *AI Safety via Debate*


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — الاتجاهات المستقبلية / الفصل 61: المحاذاة المتقدمة وسلامة الأنظمة الفائقة.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
